# CNN Reporting Notebook

Notebook này dùng để đọc artifact từ pipeline CNN mới và tạo summary/biểu đồ cho báo cáo.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import confusion_matrix

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'ai' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai.CNN.evaluator import ModelEvaluator
from ai.CNN.feature_extraction import vector_to_mbti
from infrastructure.config_loader import load_cnn_config

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)


## 2. Chọn artifact cần báo cáo

In [ ]:
RUN_NAME = 'cnn_real'

model_dir = PROJECT_ROOT / 'models' / RUN_NAME
history_path = model_dir / 'training_history.json'
metrics_path = PROJECT_ROOT / 'outputs' / 'cnn_real_metrics.json'
X_test_path = model_dir / 'X_test.npy'
y_test_path = model_dir / 'y_test.npy'
model_path = model_dir / 'audio_cnn.pt'

for path in [history_path, metrics_path, X_test_path, y_test_path, model_path]:
    print(path, 'OK' if path.exists() else 'MISSING')


## 3. Load dữ liệu báo cáo

In [ ]:
with open(history_path, 'r', encoding='utf-8') as f:
    history = json.load(f)

with open(metrics_path, 'r', encoding='utf-8') as f:
    metrics = json.load(f)

X_test = np.load(X_test_path)
y_test = np.load(y_test_path)

history_df = pd.DataFrame(history)
metrics_df = pd.DataFrame(metrics['dimensions']).T.reset_index().rename(columns={'index': 'dimension'})
summary_df = pd.DataFrame([
    {
        'overall_dimension_accuracy': metrics['overall_dimension_accuracy'],
        'overall_dimension_f1': metrics['overall_dimension_f1'],
        'full_type_accuracy': metrics['full_type_accuracy'],
        'sample_count': metrics['sample_count'],
    }
])

display(summary_df)
display(metrics_df)
print('X_test shape:', X_test.shape)
print('y_test shape:', y_test.shape)


## 4. Biểu đồ loss theo epoch

In [ ]:
ax = history_df[['train_loss', 'val_loss']].plot(marker='o')
ax.set_title(f'Training History - {RUN_NAME}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
plt.show()


## 5. Biểu đồ metric theo từng chiều MBTI

In [ ]:
plot_df = metrics_df.melt(id_vars='dimension', value_vars=['accuracy', 'f1'], var_name='metric', value_name='value')
ax = sns.barplot(data=plot_df, x='dimension', y='value', hue='metric')
ax.set_ylim(0, 1)
ax.set_title(f'Per-dimension Metrics - {RUN_NAME}')
plt.show()


## 6. Dự đoán lại trên test set và tạo confusion matrix

In [ ]:
config = load_cnn_config()
evaluator = ModelEvaluator(config)
model = evaluator.load_model(str(model_path))
probs = evaluator.predict_logits(model, X_test)
preds = (probs >= 0.5).astype(int)

dimension_names = ['E/I', 'S/N', 'T/F', 'J/P']
for idx, name in enumerate(dimension_names):
    cm = confusion_matrix(y_test[:, idx].astype(int), preds[:, idx].astype(int), labels=[0, 1])
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Confusion Matrix - {name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()


## 7. So sánh MBTI thật và MBTI dự đoán

In [ ]:
report_df = pd.DataFrame({
    'true_mbti': [vector_to_mbti(row) for row in y_test],
    'pred_mbti': [vector_to_mbti(row) for row in probs],
})
report_df['match'] = report_df['true_mbti'] == report_df['pred_mbti']
display(report_df)
print('Type-level accuracy:', report_df['match'].mean())


## 8. Gợi ý dùng cho báo cáo

- Dùng bảng summary ở trên cho phần kết quả tổng quát.
- Dùng biểu đồ loss để mô tả quá trình train.
- Dùng bar chart theo dimension để so sánh E/I, S/N, T/F, J/P.
- Dùng confusion matrix để giải thích mô hình nhầm ở đâu.